In [109]:
import numpy as np
import pandas as pd

In [110]:
# データを読み込みます
df = pd.read_csv('../Job.csv',parse_dates=['EVENT_DTM'],dtype={'SERIAL_NUMBER':'category', 'PRINT_COUNT':'int32'})

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   SERIAL_NUMBER  13 non-null     category      
 1   EVENT_DTM      13 non-null     datetime64[ns]
 2   PRINT_COUNT    13 non-null     int32         
dtypes: category(1), datetime64[ns](1), int32(1)
memory usage: 313.0 bytes


In [112]:
df['JOB_INTERVAL'] = df.EVENT_DTM.diff().map(lambda x: x.total_seconds())
df['JOB_LENGTH'] = df.PRINT_COUNT.diff()
df['SERIAL_NUMBER_SHIFT'] = df.SERIAL_NUMBER.shift()
df['CONTINUE'] = df.JOB_INTERVAL < df.JOB_LENGTH/45*60+2
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'JOB_INTERVAL'] = np.nan
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'JOB_LENGTH'] = np.nan
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'CONTINUE'] = False

In [113]:
df

,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_INTERVAL,JOB_LENGTH,SERIAL_NUMBER_SHIFT,CONTINUE
0,aaa,2020-01-01 15:15:00,5000,NaN,NaN,NaN,False
1,aaa,2020-01-02 15:15:12,5001,86412.0,1.0,aaa,False
2,aaa,2020-01-02 15:15:13,5002,1.0,1.0,aaa,True
3,aaa,2020-01-02 15:15:14,5004,1.0,2.0,aaa,True
4,aaa,2020-01-01 15:16:00,5006,-86354.0,2.0,aaa,True
5,aaa,2020-01-02 15:17:12,5007,86472.0,1.0,aaa,False
6,aaa,2020-01-02 15:18:13,5008,61.0,1.0,aaa,False
7,aaa,2020-01-03 15:18:14,5009,86401.0,1.0,aaa,False
8,aaa,2020-01-04 15:18:15,5010,86401.0,1.0,aaa,False
9,aaa,2020-01-05 15:18:16,5011,86401.0,1.0,aaa,False


In [114]:
# 連続JOB開始行のインデックスと連続JOB枚数の辞書作成
value_list = []
index_list = []
continue_dict = {}
for i, (JOB_LENGTH, CONTINUE) in enumerate(zip(df.JOB_LENGTH.values, df.CONTINUE.values)):
    if CONTINUE == False and len(index_list) == 0:
        JOB_LENGTH_BEFOR = 0
    elif CONTINUE == False and len(index_list) > 0:
        index_min = min(index_list)
        value_max = max(value_list)
        continue_dict[index_min] = value_max
        value_list = []
        index_list = []
    else:
        JOB_LENGTH += JOB_LENGTH_BEFOR
        JOB_LENGTH_BEFOR = JOB_LENGTH
        value_list.append(JOB_LENGTH)
        index = i-1
        index_list.append(index)
    
# 辞書から新しい列作成
for k, v in continue_dict.items():
    df.loc[df.index[k],'ADJUST'] = v

# 連続JOBを考慮したJOB枚数を計算
df.loc[~df.ADJUST.isnull(), 'JOB_LENGTH'] = df.JOB_LENGTH + df.ADJUST

In [115]:
df


,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_INTERVAL,JOB_LENGTH,SERIAL_NUMBER_SHIFT,CONTINUE,ADJUST
0,aaa,2020-01-01 15:15:00,5000,NaN,NaN,NaN,False,NaN
1,aaa,2020-01-02 15:15:12,5001,86412.0,6.0,aaa,False,5.0
2,aaa,2020-01-02 15:15:13,5002,1.0,1.0,aaa,True,NaN
3,aaa,2020-01-02 15:15:14,5004,1.0,2.0,aaa,True,NaN
4,aaa,2020-01-01 15:16:00,5006,-86354.0,2.0,aaa,True,NaN
5,aaa,2020-01-02 15:17:12,5007,86472.0,1.0,aaa,False,NaN
6,aaa,2020-01-02 15:18:13,5008,61.0,1.0,aaa,False,NaN
7,aaa,2020-01-03 15:18:14,5009,86401.0,1.0,aaa,False,NaN
8,aaa,2020-01-04 15:18:15,5010,86401.0,1.0,aaa,False,NaN
9,aaa,2020-01-05 15:18:16,5011,86401.0,1.0,aaa,False,NaN


In [122]:
def aaa(row):
    print(row[0])
    print(row['SERIAL_NUMBER'])
    if row[0].startswith('aa'):
        # print(row[1]+10)
        return (row[1]+10)
    else:
        # print(row[1]+100)
        return (row[1]+100)

In [123]:
# df[['A','B']] = df.apply(aaa,axis=1, result_type='expand')
# df['A'] = df[['PRINT_COUNT','JOB_LENGTH']].apply(aaa,axis=1, result_type='expand')
df['A'] = df[['SERIAL_NUMBER','JOB_LENGTH']].apply(aaa,axis=1)
# df['B'] = df[['SERIAL_NUMBER','JOB_LENGTH']].apply(aaa,axis=1)

aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
bbb
bbb
bbb
bbb
bbb
bbb


In [102]:
df

,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_INTERVAL,JOB_LENGTH,SERIAL_NUMBER_SHIFT,CONTINUE,ADJUST,A,B
0,aaa,2020-01-01 15:15:00,5000,NaN,NaN,NaN,False,NaN,NaN,NaN
1,aaa,2020-01-02 15:15:12,5001,86412.0,6.0,aaa,False,5.0,16.0,86422.0
2,aaa,2020-01-02 15:15:13,5002,1.0,1.0,aaa,True,NaN,11.0,11.0
3,aaa,2020-01-02 15:15:14,5004,1.0,2.0,aaa,True,NaN,12.0,11.0
4,aaa,2020-01-01 15:16:00,5006,-86354.0,2.0,aaa,True,NaN,12.0,-86344.0
5,aaa,2020-01-02 15:17:12,5007,86472.0,1.0,aaa,False,NaN,11.0,86482.0
6,aaa,2020-01-02 15:18:13,5008,61.0,1.0,aaa,False,NaN,11.0,71.0
7,aaa,2020-01-03 15:18:14,5009,86401.0,1.0,aaa,False,NaN,11.0,86411.0
8,aaa,2020-01-04 15:18:15,5010,86401.0,1.0,aaa,False,NaN,11.0,86411.0
9,aaa,2020-01-05 15:18:16,5011,86401.0,1.0,aaa,False,NaN,11.0,86411.0


In [617]:
# 連続Job行の削除
df = df[df.CONTINUE == False]
# 不要列の削除
df = df.drop(['CONTINUE','ADJUST','JOB_INTERVAL'], axis=1)
# JOB間隔の再計算（単位：分）
df['JOB_INTERVAL'] = round(df.EVENT_DTM.diff().map(lambda x: x.total_seconds()/60), 3)
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'JOB_INTERVAL'] = np.nan
# 不要列の削除
df = df.drop(['SERIAL_NUMBER_SHIFT'], axis=1)
df = df.dropna(subset=['JOB_LENGTH'])

In [618]:
df

,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_LENGTH,JOB_INTERVAL
1,aaa,2020-01-02 15:15:12,5001,6.0,1440.200
5,aaa,2020-01-02 15:17:12,5007,1.0,2.000
6,aaa,2020-01-02 15:18:13,5008,1.0,1.017
7,aaa,2020-01-03 15:18:14,5009,1.0,1440.017
8,aaa,2020-01-04 15:18:15,5010,1.0,1440.017
9,aaa,2020-01-05 15:18:16,5011,1.0,1440.017
12,bbb,2020-01-03 15:15:15,5020,7.0,1440.033


In [268]:
def fuser_Cycle(df):
    # 並び替え
    df = df.sort_values(['Deviceid', 'Timestamp']).reset_index(drop = True)
    # 列追加
    df['Unit'] = 'Fuser'
    df['Deviceid_Shift'] = df.Deviceid.shift()
    df['Life_Diff'] = df['Life'].diff()
    df['JOB_LENGTH'] = df['Total Page Count'].diff()
    df['Rol_Diff'] = df['Total Rol Distance'].diff()
    # Deviceidの切替りのデータをnanに置き換える
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Life_Diff'] = np.nan
    df.loc[df.Deviceid != df.Deviceid_Shift, 'JOB_LENGTH'] = np.nan
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Rol_Diff'] = np.nan
    # 交換タイミングを計算
    df.loc[(df.Life_Diff > 0) & (df.JOB_LENGTH < 0) & (df.Rol_Diff < 0), 'Event'] = '交換'
    # 不要な列を削除
    df = df.drop(['Deviceid_Shift', 'Life_Diff', 'JOB_LENGTH', 'Rol_Diff'], axis=1)
    return df

In [269]:
def life_cycle(df):
    for row in range(len(df)):
        if row == 0:
            cycle = 1
            count_new = 0
            df.loc[row,'Cycle'] = cycle
            df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount']
        else:
            # 同じ本体、新品以外
            if df.loc[row,'Event'] != '交換' and df.loc[row,'Deviceid'] == df.loc[row-1,'Deviceid']:
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
            # 同じ本体、新品
            elif df.loc[row,'Event'] == '交換' and df.loc[row,'Deviceid'] == df.loc[row-1,'Deviceid']:
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
                # 2サイクル目以降で交換後1000枚以内の再交換はnanに置き換える
                if cycle == 1:
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                elif df.loc[row,'Pagecount'] - count_new > 1000 :
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                else:
                    df.loc[row,'Event'] = np.nan
            # 違う本体への切替り、新品以外
            elif df.loc[row,'Event'] != '交換' and df.loc[row,'Deviceid'] != df.loc[row-1,'Deviceid']:
                cycle = 1
                count_new = 0
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
            # 違う本体への切替り、新品
            elif df.loc[row,'Event'] == '交換' and df.loc[row,'Deviceid'] != df.loc[row-1,'Deviceid']:
                cycle = 1
                count_new = 0
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
                # 交換後1000枚以内の再交換はnanに置き換える
                if cycle == 1:
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                elif df.loc[row,'Pagecount'] - count_new > 1000 :
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                else:
                    df.loc[row,'Event'] = np.nan
    return df

In [270]:
# 交換タイミングの取得
if apps == 'fuser_life':
    df = fuser_Cycle(df) 
elif apps == 'itb_life':
    df = fuser_Cycle(df) 

# 交換サイクルとリセットページカウントの算出
df = life_cycle(df)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,Fuser,NaN,1.0,5000.0


In [271]:
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,Fuser,NaN,1.0,5000.0


In [272]:
df['Life2'] = df.Life.apply(lambda x : x if x<5 else np.nan)

In [273]:
df = df.sort_values(['Deviceid', 'Cycle']).reset_index(drop = True)
df.groupby(['Deviceid','Cycle']).max()

/var/folders/gx/lc33vdvs5dj0sf9x1njqzryr0000gn/T/ipykernel_35050/807355777.py:2: FutureWarning: Dropping invalid columns in DataFrameGroupBy.max is deprecated. In a future version, a TypeError will be raised. Before calling .max, select only columns which should be valid for the function.
  df.groupby(['Deviceid','Cycle']).max()


Timestamp  Pagecount  Life  Total Page Count  \
Deviceid Cycle                                                 
aaa      1.0   2020-01-04      13000   100             12000   
         2.0   2020-01-06      15000   100              4000   
bbb      1.0   2020-01-08          8   100                 2   
ccc      1.0   2020-01-18      15000   100              4500   
         2.0   2020-01-21      15200   100             15000   

                Total Rol Distance   Unit  Reset_Count  Life2  
Deviceid Cycle                                                 
aaa      1.0                 12000  Fuser      13000.0    1.0  
         2.0                  4000  Fuser       2000.0    NaN  
bbb      1.0                 40000  Fuser          8.0    NaN  
ccc      1.0                  4500  Fuser      15000.0   -8.0  
         2.0                 15000  Fuser        200.0   -8.0

In [274]:
# 各デバイス毎の一番最初と最後のデータのみの日付データを作成（エラー発生なし本体の計算と、各本体のデータ取得期間確認用）
# df_serial1 = pd.DataFrame()
# df_serial2 = pd.DataFrame()
# df_serial1['start'] = df.groupby('serial').start.min()
# df_serial2['start'] = df.groupby('serial').start.max()
# df_serial = pd.concat([df_serial1,df_serial2])
# df_serial = df_serial.reset_index()
# df_serial = df_serial.sort_values(['serial','start'])
# df_serial


In [275]:
# 各デバイス毎の一番最初と最後のデータのみを抽出（エラー発生なし本体の計算と、各本体のデータ取得期間確認用）
df_group = df.groupby('serial')
df_serial1 = df.loc[df_group['start'].idxmin(),:]
df_serial2 = df.loc[df_group['start'].idxmax(),:]
df_serial = pd.concat([df_serial1,df_serial2])
df_serial = df_serial.sort_values(['serial','start'])
df_serial = df_serial.reset_index(drop=True)
df_serial

KeyError: 'serial'

In [250]:
# エラー発生時のインデックスリストを作成
index_list = df[~df.factor.isnull()].index.to_list()
index_list

AttributeError: 'DataFrame' object has no attribute 'factor'

In [276]:
# エラー発生前後50行のインデックスリストを作成
index_max = len(df)
hold_list = []
for i in index_list:
    for j in range(-50,51):
        if i+j>=0 and i+j<index_max:
            hold_list.append(i+j)
hold_list = list(set(hold_list))
hold_list

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

In [252]:
# エラー発生前後50行のデータのみを残す
df.loc[df.index[hold_list],'Hold'] = 1
df = df.dropna(subset=['Hold'])
df = df.drop('Hold',axis=1)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Life2,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,NaN,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,1.0,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,-5.0,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,NaN,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,NaN,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,NaN,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,NaN,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,NaN,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,NaN,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,-8.0,Fuser,NaN,1.0,5000.0


In [253]:
df = pd.concat([df,df_serial])
df = df.sort_values(['serial','start'])
df = df.reset_index(drop=True)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Life2,Unit,Event,Cycle,Reset_Count,start,end,nr,serial,error_code,factor,print_count
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1.0
1,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-06,2020-03-20,600-1,aaa,2.0,NaN,6.0
2,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-07,2020-01-07,701-1,bbb,4.0,NaN,7.0
3,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-15,2020-01-07,801-1,bbb,4.0,JAM,15.0
4,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-16,2020-01-07,801-1,ccc,8.0,JAM,16.0
5,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17,2020-01-07,801-1,ccc,1.0,JAM,17.0
6,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-18,2020-02-07,600-1,eee,NaN,NaN,18.0
7,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-24,2020-01-07,801-1,eee,NaN,JAM,24.0
8,aaa,2020-01-01,4000.0,74.0,4000.0,4000.0,NaN,Fuser,NaN,1.0,4000.0,NaT,NaT,NaN,NaN,NaN,NaN,NaN
9,aaa,2020-01-02,8000.0,1.0,8000.0,8000.0,1.0,Fuser,NaN,1.0,8000.0,NaT,NaT,NaN,NaN,NaN,NaN,NaN


In [4]:
print("This is a \"quote\" inside a string.")  # ダブルクォートのエスケープ
print('This is an \'example\' of single quotes.')  # シングルクォートのエスケープ
print("C:\\path\\to\\file")  # バックスラッシュのエスケープ
print("Hello\nWorld")  # 改行文字のエスケープ
print("Tab\tTab\tTab")  # タブ文字のエスケープ


This is a "quote" inside a string.
This is an 'example' of single quotes.
C:\path\to\file
Hello
World
Tab	Tab	Tab


In [ ]:
目的
プレミアム
ヒータ設計
CJの接着力判断
設計値の妥当性判断
小サイズ生産性
定着設計
MPモード設計
COCへの寄与
小サイズ生産性
定着設計
定着設計
定着設計
小サイズ生産性
固着
トナー残検レス
キャリブレーション設計
METISレス
METISレス
ループセンサ設計
制御設計
CRGトルク設計
FWランチェンの仕様
長寿命化
長寿命化
予測Dhalf設計
小サイズ生産性
小サイズ生産性
小サイズ生産性
昇温設計
METISレス
センサのコストダウン
小サイズ生産性
電気素子のコストダウン
寿命設定
設計値の妥当性判断
設計値の妥当性判断
寿命設定
寿命設定
設計簡素化
寿命設定
寿命設定
USBのコストダウン
LGTのコストダウン
電源のコストダウン
SWのコストダウン
MPモードの訴求力判断
トナー残検レス
パンチのコストダウン
FSSSの故障回避モード
電源のコストダウン
電源のコストダウン


音診断の価値判断
METISレス
ステイプルの紙種対応判断
SCNサーミスタレス
SCNの長寿命化
SCNの寿命報知
SCNの長寿命化

SCN汚れ推定
プレミアム
静音の必要性
ダンパの必要性
後端規制の必要性
斜行マージンの設計値
2BinCJの設計値
2BinCJの設計値
アクティブパンチの設計値
80ppmの必要性判断
補給CRGの配置
評価条件の適正化
マニュアルステイプルの必要性判断
CJの冊子の必要性判断
ステイプルの紙種対応判断
評価条件の適正化
FSCJ/2BinCJでの改善点判断
昇温抑制モードの必要性判断
紙シワの評価条件適正化
JAM対策？
JAM対策？
固着リスク判断
ヒータ設計
METISレス
METISレス
METIS簡素化
光学排紙センサレス
定着排紙センサ配置
定着排紙センサ配置




CCFの価値判断



左レジ調整機構のレス化
左レジ調整機構のレス化
METISレス
METISレス
METIS簡素化
METIS簡素化
トナー残検レス
除電ブラシレス
CST残量検知レス
レジ検センサレス
ITB本体括り付け
テストプリントSWレス
SCNサーミスタレス
CJの変更規模把握
CJの接着強度
CJの紙サイズ
CJの接着強度
冊子の整合性





プレミアム
CCFの価値判断
CCFの価値判断
CCFの価値判断
電源のコストダウン

10minサービスの対応




In [ ]:
調べたいこと大分類
紙補充
紙サイズ
ステイプル
ステイプル
紙サイズ
紙種
MPモード
非純正CRG
紙サイズ
FPOT
入力電圧
FPOT
紙サイズ
昇温
トナー残検
キャリブレーション
METIS
METIS
ループセンサ
非純正CRG
CRGトルク
FWバージョン
LLC交換
LLC交換
予測Dhalf
紙サイズ
紙サイズ
紙サイズ
JobLength
気圧
キャリブレーション
紙サイズ
連続通紙
排紙枚数
ステイプル
ステイプル
動作回数
動作回数
通信不良
電源ON
スリープ
USBプリント
LGT
停電警告
テストプリント
MPモード
トナー残検
パンチ
故障
IS
FMTR



紙種
ステイプル
SCNサーミスタ
SCNの実稼働状況
レディ時間の推移
SCNモータ起動の推移

レジ検
紙補充
環境音
CST強装着
後端規制ミスセット
FSSS入口斜行
ステイプル
ステイプル
FSSS入口斜行
JobLength
CRG交換
昇温
ステイプル
ステイプル
ステイプル
温湿度
JAM
昇温
紙種
紙種
印字率
環境
紙サイズ
紙種
紙種
紙種
紙種
搬送バラツキ
JAM
JAM
JAM
JAM
キャリブレーション
紙サイズ
紙の状況
紙の状況
デカーラ
左レジ調整機構
出荷時の紙サイズ位置
紙種
気圧補正



温湿度

キャリブレーション
ITB交換
テストプリント
SCNサーミスタ
JAM
紙を取るまでの時間
紙サイズ
ステイプル

紙の状態
温湿度
PV
FSSS
満検
紙補充
PV
紙サイズ
LGT

FPOT
電源の交換




In [ ]:
調べたいこと小分類
紙補充のトレイ開閉ミス
地域別の小サイズ使用状況
ステイプルの片面/両面
ステイプルのJob
スループットダウン
定着モードの指定状況
MPモードの使用状況
モータトルク
小サイズ紙の使用状況
FPOT
入力電圧
FPOT
小サイズ紙の使用状況
機内昇温の実績
トナー残検の実力
キャリブレーション頻度
METISのバラツキ
METISとの相関
ループセンサ
非純正CRG
CRGトルク
FWバージョン
LLC交換
LLC交換
予測Dhalfの実力
小サイズ紙の使用状況
小サイズ紙の使用状況
ミックス紙の使用状況
JobLength分布
気圧
キャリブレーション
小サイズ紙の使用状況
連続通紙
排紙枚数
ステイプル枚数
ステイプル回数
アクチュエータの動作回数
スイッチの動作回数
通信不良の発生状況
電源ONの使用状況
スリープの使用状況
USBプリントの使用状況
LGTの使用状況
停電警告の動作回数
テストプリントの使用状況
MPモードの使用状況
トナー残検の実力
パンチの使用状況
FSSSの故障発生状況
ISの使われ方
FMTRの負荷
サーミスタの負荷
FMTRへの出力
音診断の効果
定着モードの指定状況
ステイプルの使われ方
SCNサーミスタとの相関
SCNの実稼働状況
レディ時間の推移
SCNモータ起動の推移

レジ検の左右差
紙補充タイミング
環境音の実力
CST強装着の有無
後端規制ミスセットの有無
FSSS入口斜行の実力
2BinSSのステイプル使用状況
2BinSSのステイプル使用状況
FSSS入口斜行の実力
MPのJobLength分布
補給CRGの交換
機内昇温の実績
マニュアルステイプルの使用状況
ステイプルの使われ方
ステイプルの使われ方
設置環境
FSSS/2BinSSのJAM発生状況
昇温抑制モードの発生状況
封筒の使用実績
2BinSSのグロス紙の使用実績
2BinSSの印字率
超Hot制御の発生状況
地域別の小サイズ使用状況
定着モードの指定状況
定着モードの誤指定状況
ラフ紙の使用状況
OHTの使用状況
定着排紙センサの紙間バラツキ
定着排紙JAMの発生状況
JAMの発生状況
JAMの発生状況
アンクリJAMの発生状況
キャリブレーションの頻度
非定型の小サイズ紙の使用状況
カールの発生状況
カールの発生状況
デカーラのオプション選択
左レジ調整機構の使用状況
出荷時の紙サイズ位置
定着モードの指定状況
気圧補正の発生状況


画像パターンの実績
設置環境
紙残量検知を使用状況
キャリブレーションの頻度
ITB交換の頻度
テストプリントの使用状況
SCNサーミスタの稼働実績
FSSS/2BinSSのJAM
ユーザが紙を取るまでの時間
FSSS/2BinSSの紙サイズ
ステイプル



排紙OPとPV
ステイプル





スリープスタンバイ






In [ ]:
ボトムアップ委員
12技術開発室の上原といいます。
裾野ボトムアップ委員という活動の紹介をさせていただきます。
この活動は、裾野の本体開発で取り組んできた活動になりますが、活動の輪をさらに広げていきたいと考えているため、本日、紹介させていただくことになりました。

まずボトムアップ委員というのは、各部門のメンバーが集まって、裾野事業所における様々な改善活動をしている活動体になります。
センター員にアンケートを取ったり、目安箱やKIZUKImtgなどを通して、センター員の感じている課題を吸い上げて、自発的な取り組みを行っています。
2020年から活動をスタートしており、最初は部門選出メンバーで取り組んできましたが、昨年度から各部門の有志メンバーのこの10名で活動を継続しています。

これまでの活動の一例を紹介します。
まず、周３ちゃんねるというTeams掲示板を作成しました。
こちらは、周辺開発全体で、TeamsやOutlook等のノウハウを共有したり、分からないことを質問できる環境を提供しています。
このようなチームがあり、誰でも投稿できるようになっています。
2021年から運用を開始しており、これだけのノウハウが蓄積されてきています。
BU委員は、この場を利用してノウハウ共有や質問への回答を積極的に行っています。
また、暑い寒いシステムというものも作成しました。
こちらもすでに周辺開発全体で運用しているシステムで、空調温度に関する不満の声を施設課に自動連絡する仕組みを構築しています。
投稿数が閾値を超えると自動連絡するものになります。
2022年から運用開始しており、100回以上の温度変更を実現しています。
さらに、投稿された結果を分析して、その結果に基づいて、更なる改善提案をするというような取り組みまで実施しています。
このように、みんなのためにボランティア精神で活動を進めてきています。

ここから昨年度取り組んだ内容の紹介になります。
一つ目は、ソフトウェア管理システムの開発に取り組みました。
本体開発では、多くの部門がエクセル台帳で管理しており、様々なムダが発生していると感じていました。
それを全部門で台帳を共通化して、作業を自動化することで、様々な課題が解決されるのではないか？と考えて、プロトタイプを作成しました。
具体的には、PC台帳、ソフト台帳、ソフト管理台帳という3つの台帳を作って、インストール時には、選択して承認依頼を出すようなシステムです。
常に台帳を最新状態にできるようにすることで、ムダの解消はもちろんですが、もっと適正管理に繋がらないか？ということを考えて検討していました。
今後は管理課と連携して活動を進めていきたいと思っています。

その他にも色々な取り組みを行っています。
先ほど話した暑い寒いシステムに関しては、全社の報告会で発表し、展開を進めています。
第一開と開推で立ち上げたものを、品保、化成、CRG、そして第二開に加えて、他事業所にも展開を進めています。

また、居室電気の自動消灯という取り組みも行いました。
昼休みに人手で電気を付けたり消したりしていましたが、それを輪番制に合わせて自動消灯できるように、施設課と管理課と連携して実現させました。

物品持ち出し管理システムの展開という取り組みも行いました。
電気部門でRPA化したシステムを使いたいという声が上がってきたので、他部門に展開する取り組みも行いました。

最後は、遊び心で取り組んだものになりますが、感謝の気持ちを伝えるアプリというものも開発しました。
全社の有志メンバーと一緒にアプリを開発しました。
感謝の気持ちを公開できるようにすることで、ほめあう文化の醸成に繋げていきたいという狙いです。

働きやすい環境や効率化に向けて、今年もさらに、いろんな取り組みを継続していきたいと思っています。
暑い寒いシステムを進化させるために、温度センサを使って自動投稿できるようにしたら、もっと快適になるのではないか？とか、
社内のマッチングアプリを作って、スキルや興味の合うメンバーでの勉強会に繋げたらどうか？とか、日々、色々考えています。
活動メンバーは随時募集しています。
お気軽にボトムアップ委員にお声がけいただきたいです。
以上となります。ありがとうございました。



